In [1]:
# ============================================================
# FCM-PSO (ED-UC-VPE-POZ) + Ramp Enforcement (WITH STARTUP FIX)
# UPDATED FULL CODE:
# - Writes BOTH:
#   1) "Schedulling_Raw"      : pre ramp-enforcement (stage-1)
#   2) "Schedulling_Adjusted" : post ramp-enforcement (consistent with Ramp Rates)
# - Adds POZ-aware static UC escalation (prevents "Infeasible in bounds" at stage-1)
# - FIX: Startup-aware ramp enforcement for units that were OFF -> ON
#        (prevents perpetual ramp violations when ramp_up < pmin)
# ============================================================

import pandas as pd
import numpy as np
import time
import itertools
import skfuzzy as fuzz
from pyswarm import pso
import re

# ============================================================
# CONFIG: file paths
# ============================================================
UNIT_DATA_PATH = "data/ELD_Test_System.xlsx"
DEMAND_DATA_PATH = "data/powerdemand.xlsx"
OUTPUT_FILE_PATH = "FCM-PSO (ED-UC-VPE-POZ).xlsx"

# ============================================================
# USER SETTINGS
# ============================================================
DEMAND_TOL = 1e-6
POZ_EPS = 1e-6
penalty_factor = 1e8

N_CLUSTERS = 3
FCM_MIN_SAMPLES = 10

PSO_SWARMSIZE = 50
PSO_MAXITER = 200

# ============================================================
# Load Dataset
# ============================================================
data = pd.read_excel(UNIT_DATA_PATH)

a = data["a"].values.astype(float)
b = data["b"].values.astype(float)
c = data["c"].values.astype(float)

d = data["d"].values.astype(float)
e = data["e"].values.astype(float)

Minimum_Capacity = data["pmin"].values.astype(float)
Maximum_Capacity = data["pmax"].values.astype(float)

ramp_up = data["rampup"].values.astype(float)
ramp_down = data["rampdown"].values.astype(float)

Unit = data["Unit"].values

poz_raw = data["poz"].values if "poz" in data.columns else np.array([None] * len(Unit), dtype=object)

power_demand_data = pd.read_excel(DEMAND_DATA_PATH)
power_demand = power_demand_data["load"].values.astype(float)

# ============================================================
# POZ parsing utilities
# ============================================================
def parse_poz_cell(cell):
    if cell is None:
        return []
    if isinstance(cell, float) and np.isnan(cell):
        return []
    s = str(cell).strip()
    if s == "" or s.lower() in ["nan", "none", "null", "[]"]:
        return []

    nums = re.findall(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?", s)
    vals = [float(x) for x in nums]
    if len(vals) < 2:
        return []

    intervals = []
    for i in range(0, len(vals) - 1, 2):
        lo, hi = vals[i], vals[i + 1]
        if hi < lo:
            lo, hi = hi, lo
        intervals.append((lo, hi))

    intervals.sort(key=lambda x: x[0])
    merged = []
    for lo, hi in intervals:
        if not merged or lo > merged[-1][1]:
            merged.append([lo, hi])
        else:
            merged[-1][1] = max(merged[-1][1], hi)
    return [(x[0], x[1]) for x in merged]

POZ_INTERVALS = [parse_poz_cell(cell) for cell in poz_raw]

def in_any_poz(p, intervals):
    for lo, hi in intervals:
        if lo <= p <= hi:
            return True
    return False

def project_out_of_poz_with_bounds(p, intervals, lb, ub, eps=POZ_EPS):
    p0 = float(np.clip(p, lb, ub))
    if not intervals:
        return p0
    if not in_any_poz(p0, intervals):
        return p0

    candidates = [lb, ub]
    for lo, hi in intervals:
        candidates += [lo - eps, hi + eps, lo, hi]

    feasible = []
    for x in candidates:
        x = float(np.clip(x, lb, ub))
        if not in_any_poz(x, intervals):
            feasible.append(x)

    if feasible:
        return min(feasible, key=lambda x: abs(x - p0))
    return p0

def apply_poz_vector_with_bounds(P, lb_vec, ub_vec, commitment):
    P = np.asarray(P, dtype=float).copy()
    lb_vec = np.asarray(lb_vec, dtype=float)
    ub_vec = np.asarray(ub_vec, dtype=float)
    commitment = np.asarray(commitment, dtype=int)

    for i in range(len(P)):
        if commitment[i] == 0:
            P[i] = 0.0
        else:
            P[i] = project_out_of_poz_with_bounds(P[i], POZ_INTERVALS[i], lb_vec[i], ub_vec[i])
    return P

def apply_poz_vector(P, commitment):
    lb_vec = Minimum_Capacity * commitment
    ub_vec = Maximum_Capacity * commitment
    return apply_poz_vector_with_bounds(P, lb_vec, ub_vec, commitment)

# ============================================================
# POZ-aware static feasibility helpers (prevents stage-1 infeasible)
# ============================================================
def max_feasible_not_in_poz(intervals, lb, ub, eps=POZ_EPS):
    lb = float(lb)
    ub = float(ub)
    if ub < lb:
        ub = lb

    if (not intervals) or (not in_any_poz(ub, intervals)):
        return ub

    cand = [lb, ub]
    for lo, hi in intervals:
        cand.append(float(np.clip(lo - eps, lb, ub)))
        cand.append(float(np.clip(hi + eps, lb, ub)))

    feas = []
    for x in cand:
        x = float(np.clip(x, lb, ub))
        if not in_any_poz(x, intervals):
            feas.append(x)

    if feas:
        return max(feas)

    xproj = project_out_of_poz_with_bounds(ub, intervals, lb, ub, eps=eps)
    return float(np.clip(xproj, lb, ub))

def max_feasible_total_static_with_poz(commitment):
    commitment = np.asarray(commitment, dtype=int)
    total = 0.0
    for i in range(len(commitment)):
        if commitment[i] == 0:
            continue
        lb = Minimum_Capacity[i]
        ub = Maximum_Capacity[i]
        ub_eff = max_feasible_not_in_poz(POZ_INTERVALS[i], lb, ub)
        total += ub_eff
    return float(total)

def escalate_uc_for_static_feasibility_poz(demand, base_commitment, max_add=None):
    demand = float(demand)
    commitment = np.asarray(base_commitment, dtype=int).copy()

    if max_feasible_total_static_with_poz(commitment) >= demand - DEMAND_TOL:
        return commitment

    off = np.where(commitment == 0)[0].tolist()
    if not off:
        return commitment

    scores = []
    for i in off:
        lb = Minimum_Capacity[i]
        ub = Maximum_Capacity[i]
        ub_eff = max_feasible_not_in_poz(POZ_INTERVALS[i], lb, ub)
        gain = max(0.0, ub_eff)
        scores.append((gain, i))

    scores.sort(reverse=True, key=lambda x: x[0])

    added = 0
    for gain, i in scores:
        if gain <= 0:
            continue
        commitment[i] = 1
        added += 1
        if max_feasible_total_static_with_poz(commitment) >= demand - DEMAND_TOL:
            return commitment
        if max_add is not None and added >= max_add:
            break

    return commitment

# ============================================================
# CHECKERS
# ============================================================
def vpe_checker(P, commitment):
    P = np.asarray(P, dtype=float)
    commitment = np.asarray(commitment, dtype=int)
    vpe_term = np.abs(d * np.sin(e * (Minimum_Capacity - P)))
    vpe_term = np.where(commitment == 1, vpe_term, 0.0)
    active = np.where(commitment == 1, np.abs(vpe_term) > 1e-12, False)
    return vpe_term, float(np.sum(vpe_term)), active

def poz_checker(P, commitment):
    P = np.asarray(P, dtype=float)
    commitment = np.asarray(commitment, dtype=int)
    violation = np.zeros_like(P, dtype=bool)
    which_interval = [None] * len(P)

    for i in range(len(P)):
        if commitment[i] == 0:
            continue
        for (lo, hi) in POZ_INTERVALS[i]:
            if lo <= P[i] <= hi:
                violation[i] = True
                which_interval[i] = (lo, hi)
                break

    return violation, which_interval, int(np.sum(violation))

# ============================================================
# COST helpers + OBJECTIVE
# ============================================================
def base_cost_per_unit(P_full):
    P_full = np.asarray(P_full, dtype=float)
    return (a + b * P_full + c * (P_full ** 2))

def objective_function_full(P_full, demand, commitment):
    P_full = np.asarray(P_full, dtype=float)
    commitment = np.asarray(commitment, dtype=int)

    base_cost = np.sum((a + b * P_full + c * P_full**2))

    vpe_term = np.abs(d * np.sin(e * (Minimum_Capacity - P_full)))
    vpe_term = np.where(commitment == 1, vpe_term, 0.0)
    vpe_cost = np.sum(vpe_term)

    total_power = float(np.sum(P_full))
    deficit = max(0.0, float(demand) - total_power)
    penalty_power = penalty_factor * (deficit ** 2)

    penalty_poz = 0.0
    poz_violation, poz_interval, _ = poz_checker(P_full, commitment)
    for i in range(len(P_full)):
        if poz_violation[i]:
            lo, hi = poz_interval[i]
            dist = min(abs(P_full[i] - lo), abs(hi - P_full[i]))
            penalty_poz += 1e7 + 1e5 * dist

    return base_cost + vpe_cost + penalty_power + penalty_poz

# ============================================================
# UC helpers (basic + ramp-based escalation in post stage)
# ============================================================
def apply_unit_commitment_basic(demand):
    pmin = Minimum_Capacity
    pmax = Maximum_Capacity
    if demand < np.sum(pmin):
        for r in range(1, len(pmin) + 1):
            for subset in itertools.combinations(range(len(pmin)), r):
                active = np.zeros(len(pmin), dtype=int)
                active[list(subset)] = 1
                if np.sum(pmin[active == 1]) <= demand <= np.sum(pmax[active == 1]):
                    return active
    return np.ones(len(pmin), dtype=int)

def max_feasible_total(prev_P, commitment):
    prev_P = np.asarray(prev_P, dtype=float)
    commitment = np.asarray(commitment, dtype=int)
    ub = np.zeros_like(prev_P, dtype=float)
    for i in range(len(prev_P)):
        if commitment[i] == 0:
            ub[i] = 0.0
        else:
            ub[i] = min(Maximum_Capacity[i], prev_P[i] + ramp_up[i])
    return float(np.sum(ub))

def escalate_uc_for_feasibility(prev_P, demand, base_commitment):
    commitment = base_commitment.copy().astype(int)
    prev_P = np.asarray(prev_P, dtype=float)

    if max_feasible_total(prev_P, commitment) >= float(demand) - DEMAND_TOL:
        return commitment

    off = np.where(commitment == 0)[0].tolist()
    if not off:
        return commitment

    scores = []
    for i in off:
        prev_i = 0.0
        ub_i = min(Maximum_Capacity[i], prev_i + ramp_up[i])
        gain = 0.0 if ub_i < Minimum_Capacity[i] - 1e-9 else ub_i
        scores.append((gain, i))

    scores.sort(reverse=True, key=lambda x: x[0])

    for gain, i in scores:
        if gain <= 0:
            continue
        commitment[i] = 1
        if max_feasible_total(prev_P, commitment) >= float(demand) - DEMAND_TOL:
            return commitment

    return commitment

# ============================================================
# FCM
# ============================================================
def compute_fcm_centers(history_P_full, n_clusters=3, m=2, error=0.005, maxiter=1000):
    X = np.asarray(history_P_full, dtype=float).T
    cntr, u, _, _, _, _, _ = fuzz.cluster.cmeans(
        data=X, c=n_clusters, m=m, error=error, maxiter=maxiter, init=None
    )
    return cntr, u

def pick_init_from_centers(centers, reference_P):
    reference_P = np.asarray(reference_P, dtype=float)
    dists = np.linalg.norm(centers - reference_P[None, :], axis=1)
    return centers[int(np.argmin(dists))].copy()

# ============================================================
# PSO (optimize only ON units)
# ============================================================
def optimize_with_pso_pyswarm(demand, commitment, init_P_full=None):
    commitment = np.asarray(commitment, dtype=int)
    on_idx = np.where(commitment == 1)[0]
    if len(on_idx) == 0:
        return np.zeros_like(commitment, dtype=float), np.inf

    lb = Minimum_Capacity[on_idx].astype(float)
    ub = Maximum_Capacity[on_idx].astype(float)
    eps = 1e-9
    ub = np.where(ub <= lb, lb + eps, ub)

    def obj_on(P_on):
        P_full = np.zeros(len(commitment), dtype=float)
        P_full[on_idx] = P_on
        P_full = apply_poz_vector(P_full, commitment)
        return objective_function_full(P_full, demand, commitment)

    best_x_on, best_val = None, np.inf
    if init_P_full is not None:
        init_P_full = np.asarray(init_P_full, dtype=float)
        init_P_full = np.clip(init_P_full, Minimum_Capacity * commitment, Maximum_Capacity * commitment)
        init_P_full = apply_poz_vector(init_P_full, commitment)
        init_on = np.clip(init_P_full[on_idx].copy(), lb, ub)
        val = obj_on(init_on)
        best_x_on, best_val = init_on.copy(), val

    xopt_on, fopt = pso(
        obj_on, lb, ub,
        swarmsize=PSO_SWARMSIZE,
        maxiter=PSO_MAXITER,
        debug=False
    )

    if best_x_on is not None and best_val < fopt:
        xopt_on, fopt = best_x_on, best_val

    P_full = np.zeros(len(commitment), dtype=float)
    P_full[on_idx] = xopt_on
    P_full = apply_poz_vector(P_full, commitment)
    return P_full, float(fopt)

# ============================================================
# Demand repair inside bounds
# ============================================================
def repair_to_meet_demand_with_bounds(P, demand, lb_vec, ub_vec, commitment, tol=DEMAND_TOL, max_iters=60):
    P = np.asarray(P, dtype=float).copy()
    lb_vec = np.asarray(lb_vec, dtype=float)
    ub_vec = np.asarray(ub_vec, dtype=float)
    commitment = np.asarray(commitment, dtype=int)

    P = np.clip(P, lb_vec, ub_vec)
    P = apply_poz_vector_with_bounds(P, lb_vec, ub_vec, commitment)

    for _ in range(max_iters):
        total = float(np.sum(P))
        if total >= float(demand) - tol:
            break

        deficit = float(demand) - total
        headroom = (ub_vec - P) * commitment
        headroom = np.maximum(headroom, 0.0)
        hsum = float(np.sum(headroom))
        if hsum <= tol:
            break

        inc = deficit * (headroom / hsum)
        P = P + inc
        P = np.clip(P, lb_vec, ub_vec)
        P = apply_poz_vector_with_bounds(P, lb_vec, ub_vec, commitment)

    total = float(np.sum(P))
    if total < float(demand) - tol:
        deficit = float(demand) - total
        headroom = (ub_vec - P) * commitment
        headroom = np.maximum(headroom, 0.0)
        if float(np.sum(headroom)) > tol:
            j = int(np.argmax(headroom))
            P[j] = min(P[j] + deficit, ub_vec[j])
            P = np.clip(P, lb_vec, ub_vec)
            P = apply_poz_vector_with_bounds(P, lb_vec, ub_vec, commitment)

    if float(np.sum(P)) < float(demand) - tol:
        raise ValueError(f"Infeasible in bounds: Total {float(np.sum(P))} < Demand {demand}.")

    return P

# ============================================================
# RAMP + POZ + DEMAND enforcement (STARTUP-AWARE FIX)
# ============================================================
def enforce_ramp_poz_and_demand(prev_P, curr_P, demand, commitment, prev_commitment):
    """
    Startup-aware:
    - If unit was OFF -> ON, allow output in [0, ramp_up] for that period (do NOT force pmin).
    - If unit was ON -> ON, enforce [max(pmin, prev-ramp_down), min(pmax, prev+ramp_up)].
    """
    prev = np.asarray(prev_P, dtype=float)
    cur = np.asarray(curr_P, dtype=float)
    commitment = np.asarray(commitment, dtype=int)
    prev_commitment = np.asarray(prev_commitment, dtype=int)

    lb = np.zeros_like(cur, dtype=float)
    ub = np.zeros_like(cur, dtype=float)

    for i in range(len(cur)):
        if commitment[i] == 0:
            lb[i] = 0.0
            ub[i] = 0.0
            continue

        was_off = (prev_commitment[i] == 0)
        if was_off:
            # STARTUP: do not force pmin in the first period after turning on
            lb[i] = 0.0
            ub[i] = min(Maximum_Capacity[i], 0.0 + ramp_up[i])
            if ub[i] < lb[i]:
                ub[i] = lb[i]
        else:
            lb[i] = max(Minimum_Capacity[i], prev[i] - ramp_down[i])
            ub[i] = min(Maximum_Capacity[i], prev[i] + ramp_up[i])
            if ub[i] < lb[i]:
                ub[i] = lb[i]

    P = np.clip(cur, lb, ub)
    P = apply_poz_vector_with_bounds(P, lb, ub, commitment)
    P = repair_to_meet_demand_with_bounds(P, demand, lb, ub, commitment)
    P = np.clip(P, lb, ub)
    P = apply_poz_vector_with_bounds(P, lb, ub, commitment)
    return P

# ============================================================
# Helper: build sched row
# ============================================================
def build_sched_row_from_arrays(demand, uc, P, computation_time=None):
    P = np.asarray(P, dtype=float)
    uc = np.asarray(uc, dtype=int)

    base_vec = base_cost_per_unit(P)
    vpe_vec, _, _ = vpe_checker(P, uc)
    total_vec = base_vec + vpe_vec
    poz_v, poz_int, poz_n = poz_checker(P, uc)

    row = {"Demand": float(demand)}
    for i, u in enumerate(Unit):
        row[f"Unit {u} Status"] = int(uc[i])
        row[f"Unit {u} Power Produced"] = float(P[i])
        row[f"Unit {u} Base Cost"] = float(base_vec[i])
        row[f"Unit {u} VPE Ripple"] = float(vpe_vec[i])
        row[f"Unit {u} Total Cost"] = float(total_vec[i])
        row[f"Unit {u} POZViolation"] = bool(poz_v[i])
        row[f"Unit {u} POZInterval"] = str(poz_int[i]) if poz_int[i] is not None else ""

    total_power = float(np.sum(P))
    row["Total Power Produced"] = float(total_power)
    row["Feasible (P>=D)"] = bool(total_power >= float(demand) - DEMAND_TOL)
    row["POZ Violations Count"] = int(poz_n)
    if computation_time is not None:
        row["Computation Time"] = float(computation_time)
    return row

# ============================================================
# Main loop (Stage-1 optimization)
# ============================================================
all_outputs = []
schedulling_raw = []
history_P = []
last_P = None

for t, demand in enumerate(power_demand, start=1):
    start_time = time.time()

    # base UC
    uc = apply_unit_commitment_basic(demand)

    # POZ-aware static UC escalation (prevents stage-1 infeasible)
    uc = escalate_uc_for_static_feasibility_poz(demand, uc)

    # FCM init
    init_P = None
    if last_P is not None and len(history_P) >= FCM_MIN_SAMPLES:
        try:
            centers, _ = compute_fcm_centers(history_P, n_clusters=N_CLUSTERS)
            init_P = pick_init_from_centers(centers, last_P)
        except Exception:
            init_P = None

    # PSO optimize
    P, _ = optimize_with_pso_pyswarm(demand, uc, init_P_full=init_P)

    # demand enforce (static bounds)
    lb_full = Minimum_Capacity * uc
    ub_full = Maximum_Capacity * uc
    P = repair_to_meet_demand_with_bounds(P, demand, lb_full, ub_full, uc)

    # compute costs
    base_vec = base_cost_per_unit(P)
    vpe_vec, _, _ = vpe_checker(P, uc)
    total_vec = base_vec + vpe_vec
    poz_v, poz_int, poz_n = poz_checker(P, uc)

    total_power = float(np.sum(P))
    comp_time = time.time() - start_time

    output = pd.DataFrame({
        "Unit": Unit,
        "On/Off Status": uc,
        "Power Produced (MW)": P,
        "Base Cost (Rp equiv)": base_vec,
        "VPE Ripple (Rp equiv)": vpe_vec,
        "Total Cost (Rp equiv)": total_vec,
        "POZ": [POZ_INTERVALS[i] for i in range(len(Unit))],
        "POZ Violation": poz_v,
        "POZ Violating Interval": [str(x) if x is not None else "" for x in poz_int],
    })

    all_outputs.append({
        "Demand": float(demand),
        "Output": output,
        "Total Power Produced": total_power,
        "Total Base Cost": float(np.sum(base_vec)),
        "Total VPE Ripple": float(np.sum(vpe_vec)),
        "Total Cost": float(np.sum(total_vec)),
        "Computation Time": float(comp_time),
        "Unit Commitment": uc.copy(),
    })

    schedulling_raw.append(build_sched_row_from_arrays(demand, uc, P, computation_time=comp_time))

    last_P = P.copy()
    history_P.append(P.copy())

# ============================================================
# SAVE + RAMP ENFORCEMENT (updates all_outputs to adjusted truth)
# ============================================================
with pd.ExcelWriter(OUTPUT_FILE_PATH) as writer:
    summary_rows = []
    ramp_rates = []
    generation_limits = []

    for item in all_outputs:
        summary_rows.append({
            "Demand": item["Demand"],
            "Total Power Produced": item["Total Power Produced"],
            "Feasible (P>=D)": bool(item["Total Power Produced"] >= item["Demand"] - DEMAND_TOL),
            "Total Base Cost": item["Total Base Cost"],
            "Total VPE Ripple": item["Total VPE Ripple"],
            "Total Cost": item["Total Cost"],
            "Computation Time": item["Computation Time"],
        })

    for i in range(1, len(all_outputs)):
        prev = all_outputs[i - 1]
        curr = all_outputs[i]
        demand = float(curr["Demand"])

        prev_P = prev["Output"]["Power Produced (MW)"].values.astype(float)
        curr_P = curr["Output"]["Power Produced (MW)"].values.astype(float)

        prev_uc = prev["Unit Commitment"].copy()
        uc = curr["Unit Commitment"].copy()

        # ramp-based escalation (post)
        uc = escalate_uc_for_feasibility(prev_P, demand, uc)

        # new ON units start from 0
        curr_P2 = curr_P.copy()
        for k in range(len(curr_P2)):
            if curr["Unit Commitment"][k] == 0 and uc[k] == 1:
                curr_P2[k] = 0.0

        # STARTUP-AWARE enforcement (pass prev_uc)
        adjusted = enforce_ramp_poz_and_demand(prev_P, curr_P2, demand, uc, prev_uc)

        # checks
        ramp_rate = adjusted - prev_P
        ramp_viol = np.zeros_like(adjusted, dtype=bool)
        for k in range(len(adjusted)):
            if uc[k] == 0:
                continue
            # startup: skip ramp violation check (or define separate startup limits if you want)
            if prev_uc[k] == 0 and uc[k] == 1:
                ramp_viol[k] = False
                continue
            ramp_viol[k] = (ramp_rate[k] > ramp_up[k] + 1e-9) or (ramp_rate[k] < -ramp_down[k] - 1e-9)

        gen_viol = np.zeros_like(adjusted, dtype=bool)
        for k in range(len(adjusted)):
            if uc[k] == 0:
                gen_viol[k] = (adjusted[k] != 0.0)
                continue

            # startup interval: allow below pmin (0..ramp_up)
            if prev_uc[k] == 0 and uc[k] == 1:
                gen_viol[k] = (adjusted[k] < 0.0 - 1e-9) or (adjusted[k] > Maximum_Capacity[k] + 1e-9)
            else:
                gen_viol[k] = (adjusted[k] < Minimum_Capacity[k] - 1e-9) or (adjusted[k] > Maximum_Capacity[k] + 1e-9)

        vpe_vec_raw, _, vpe_active = vpe_checker(adjusted, uc)
        poz_v, poz_int, poz_n = poz_checker(adjusted, uc)

        total_power = float(np.sum(adjusted))
        feasible = bool(total_power >= demand - DEMAND_TOL)

        rr = {
            "Demand": demand,
            "Total Power Produced": total_power,
            "Feasible (P>=D)": feasible,
            "VPE Total": float(np.sum(vpe_vec_raw)),
            "Any VPE Active": bool(np.any(vpe_active)),
            "POZ Violations Count": int(poz_n),
            "Any POZ Violation": bool(poz_n > 0),
        }
        gl = rr.copy()

        for idx, u in enumerate(Unit):
            rr[f"Unit {u} RampRate"] = float(ramp_rate[idx])
            rr[f"Unit {u} RampViolation"] = bool(ramp_viol[idx])
            rr[f"Unit {u} VPE"] = float(vpe_vec_raw[idx])
            rr[f"Unit {u} POZViolation"] = bool(poz_v[idx])
            rr[f"Unit {u} POZInterval"] = str(poz_int[idx]) if poz_int[idx] is not None else ""

            gl[f"Unit {u} Power"] = float(adjusted[idx])
            gl[f"Unit {u} GenLimitViolation"] = bool(gen_viol[idx])
            gl[f"Unit {u} VPE"] = float(vpe_vec_raw[idx])
            gl[f"Unit {u} POZViolation"] = bool(poz_v[idx])
            gl[f"Unit {u} POZInterval"] = str(poz_int[idx]) if poz_int[idx] is not None else ""

        ramp_rates.append(rr)
        generation_limits.append(gl)

        # update truth
        all_outputs[i]["Unit Commitment"] = uc.copy()
        all_outputs[i]["Output"]["On/Off Status"] = uc.copy()
        all_outputs[i]["Output"]["Power Produced (MW)"] = adjusted

        base_vec = base_cost_per_unit(adjusted)
        vpe_vec = vpe_vec_raw
        total_vec = base_vec + vpe_vec

        all_outputs[i]["Output"]["Base Cost (Rp equiv)"] = base_vec
        all_outputs[i]["Output"]["VPE Ripple (Rp equiv)"] = vpe_vec
        all_outputs[i]["Output"]["Total Cost (Rp equiv)"] = total_vec

        all_outputs[i]["Total Power Produced"] = total_power
        all_outputs[i]["Total Base Cost"] = float(np.sum(base_vec))
        all_outputs[i]["Total VPE Ripple"] = float(np.sum(vpe_vec))
        all_outputs[i]["Total Cost"] = float(np.sum(total_vec))

        summary_rows[i]["Total Power Produced"] = total_power
        summary_rows[i]["Feasible (P>=D)"] = feasible
        summary_rows[i]["Total Base Cost"] = all_outputs[i]["Total Base Cost"]
        summary_rows[i]["Total VPE Ripple"] = all_outputs[i]["Total VPE Ripple"]
        summary_rows[i]["Total Cost"] = all_outputs[i]["Total Cost"]

    # Build adjusted schedulling from updated all_outputs
    schedulling_adjusted = []
    for item in all_outputs:
        demand = float(item["Demand"])
        uc = item["Unit Commitment"].copy()
        P_adj = item["Output"]["Power Produced (MW)"].values.astype(float)
        schedulling_adjusted.append(build_sched_row_from_arrays(demand, uc, P_adj, computation_time=item["Computation Time"]))

    pd.DataFrame(schedulling_raw).to_excel(writer, sheet_name="Schedulling_Raw", index=False)
    pd.DataFrame(schedulling_adjusted).to_excel(writer, sheet_name="Schedulling_Adjusted", index=False)
    pd.DataFrame(summary_rows).to_excel(writer, sheet_name="Summary", index=False)

    if ramp_rates:
        pd.DataFrame(ramp_rates).to_excel(writer, sheet_name="Ramp Rates", index=False)
    if generation_limits:
        pd.DataFrame(generation_limits).to_excel(writer, sheet_name="Generation Limits", index=False)

print(f"All results saved to '{OUTPUT_FILE_PATH}'")

Stopping search: Swarm best position change less than 1e-08
Stopping search: Swarm best objective change less than 1e-08
Stopping search: Swarm best position change less than 1e-08
Stopping search: Swarm best objective change less than 1e-08
Stopping search: Swarm best position change less than 1e-08
Stopping search: Swarm best objective change less than 1e-08
Stopping search: Swarm best position change less than 1e-08
Stopping search: Swarm best objective change less than 1e-08
Stopping search: Swarm best objective change less than 1e-08
Stopping search: Swarm best objective change less than 1e-08
Stopping search: Swarm best objective change less than 1e-08
Stopping search: Swarm best position change less than 1e-08
Stopping search: maximum iterations reached --> 200
Stopping search: Swarm best objective change less than 1e-08
Stopping search: Swarm best objective change less than 1e-08
Stopping search: maximum iterations reached --> 200
Stopping search: Swarm best position change les